# Modul 08: Klassifikation mit NumPy, Clustering und PCA | Lösungen

## Überblick

Sie berechnen Logits, Wahrscheinlichkeiten und Kreuzentropie, trainieren eine logistische Regression mit NumPy und vergleichen einfache Klassifikatoren. Danach implementieren Sie zentrale K-Means- und PCA-Schritte und beurteilen Clusterqualität, Skalierung und Anomaliehinweise.

**Zugehörige Vorlesungen**

- **Klassifikation mit NumPy**
- **Cluster und PCA**

## Lernziele

Nach der Bearbeitung können Sie:

- binäre Klassifikationsscores, Sigmoid-Wahrscheinlichkeiten, Schwellen und Kreuzentropie berechnen.
- eine kleine logistische Regression mit vektorisierten NumPy-Gradienten trainieren und mit Baselines vergleichen.
- K-Means-Zuordnung und Zentrenaktualisierung sowie PCA-Projektion und erklärte Varianz nachvollziehen.

## Geprüfte Fähigkeiten

- Logits, Sigmoid, binäre Kreuzentropie und Entscheidungsgrenze
- K-Means-Distanzen, Inertia, Silhouette und Gegenbeispiele
- Kovarianz, Eigenvektoren, PCA-Projektion und Anomaliehinweise

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** mittel bis anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Kleine synthetische Datensätze ermöglichen manuelle Berechnungen und schnelle Modellvergleiche. Alle Zufallsprozesse sind fest initialisiert.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import load_iris, make_blobs, make_classification, make_moons
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

X_bin, y_bin = make_classification(
    n_samples=220,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=RANDOM_SEED,
)

X_cluster, y_cluster_true = make_blobs(
    n_samples=180,
    centers=[(-3, -2), (0, 3), (3, -1)],
    cluster_std=[0.7, 0.8, 0.65],
    random_state=RANDOM_SEED,
)

print("Einrichtung abgeschlossen.")
print("Binäre Klassifikation:", X_bin.shape)
print("Clustering:", X_cluster.shape)

### Aufgabe 1: Logits, Sigmoid, Schwellen und Kreuzentropie

1. Implementieren Sie eine numerisch stabile `sigmoid`-Funktion.
2. Berechnen Sie für vorgegebene Merkmale und Gewichte Logits und Wahrscheinlichkeiten.
3. Erzeugen Sie Labels für Schwellen 0,50 und 0,70.
4. Implementieren Sie die mittlere binäre Kreuzentropie.
5. Vergleichen Sie zwei Wahrscheinlichkeitsvektoren mit gleicher Trefferzahl, aber unterschiedlicher Sicherheit.

In [ ]:
X_klein = np.array([[0.2, 1.1], [1.0, 0.5], [1.8, 1.4], [-0.8, -1.0]], dtype=float)
y_klein = np.array([1, 1, 1, 0], dtype=int)
gewichte = np.array([1.3, 0.9], dtype=float)
bias = -0.6


def sigmoid(logits):
    pass


def binaere_kreuzentropie(y_true, wahrscheinlichkeiten):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def sigmoid(logits):
    # Begrenzung schützt exp() vor extremen Überläufen in diesem Lernbeispiel.
    z = np.clip(np.asarray(logits, dtype=float), -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def binaere_kreuzentropie(y_true, wahrscheinlichkeiten):
    y_true = np.asarray(y_true, dtype=float)
    p = np.clip(np.asarray(wahrscheinlichkeiten, dtype=float), 1e-12, 1 - 1e-12)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))


logits = X_klein @ gewichte + bias
wahrscheinlichkeiten = sigmoid(logits)
labels_050 = (wahrscheinlichkeiten >= 0.50).astype(int)
labels_070 = (wahrscheinlichkeiten >= 0.70).astype(int)

print("Logits:", np.round(logits, 3))
print("Wahrscheinlichkeiten:", np.round(wahrscheinlichkeiten, 3))
print("Labels bei 0,50:", labels_050)
print("Labels bei 0,70:", labels_070)
print("Kreuzentropie:", binaere_kreuzentropie(y_klein, wahrscheinlichkeiten))

# Beide Vektoren können dieselben harten Labels liefern, aber unterschiedliche Qualität besitzen.
p_vorsichtig = np.array([0.60, 0.58, 0.62, 0.40])
p_sicher = np.array([0.92, 0.88, 0.95, 0.08])
print("Verlust vorsichtig:", binaere_kreuzentropie(y_klein, p_vorsichtig))
print("Verlust sicher und korrekt:", binaere_kreuzentropie(y_klein, p_sicher))

> **Musterantwort und Interpretation**
>
> Für die wahre Klasse wird der negative Logarithmus ihrer vorhergesagten Wahrscheinlichkeit verwendet. Nähert sich diese Wahrscheinlichkeit null, wächst der Verlust sehr stark. Das Modell wird daher nicht nur für das falsche Label, sondern zusätzlich für unbegründete Sicherheit bestraft.

### Aufgabe 2: Logistische Regression vollständig mit NumPy trainieren

1. Teilen Sie `X_bin` stratifiziert in Training und Test.
2. Standardisieren Sie leakage-frei.
3. Implementieren Sie `trainiere_logistische_regression` mit Gewichten, Bias, Kreuzentropie und vektorisierten Gradienten.
4. Trainieren Sie 800 Epochen mit einer geeigneten Lernrate.
5. Visualisieren Sie den Verlustverlauf und berechnen Sie Testgenauigkeit.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bin,
    y_bin,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_bin,
)

skalierer_bin = StandardScaler()
X_train_s = skalierer_bin.fit_transform(X_train)
X_test_s = skalierer_bin.transform(X_test)


def trainiere_logistische_regression(X, y, lernrate=0.1, epochen=800):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def trainiere_logistische_regression(X, y, lernrate=0.1, epochen=800):
    anzahl_beispiele, anzahl_merkmale = X.shape
    w = np.zeros(anzahl_merkmale, dtype=float)
    b = 0.0
    verlauf = []

    for epoche in range(epochen):
        logits = X @ w + b
        p = sigmoid(logits)

        # Wahrscheinlichkeitsfehler ist der gemeinsame Kern beider Gradienten.
        fehler = p - y
        grad_w = (X.T @ fehler) / anzahl_beispiele
        grad_b = np.mean(fehler)

        w -= lernrate * grad_w
        b -= lernrate * grad_b

        if epoche % 10 == 0 or epoche == epochen - 1:
            verlauf.append((epoche, binaere_kreuzentropie(y, p)))

    return w, b, pd.DataFrame(verlauf, columns=["Epoche", "Kreuzentropie"])


w_numpy, b_numpy, historie_numpy = trainiere_logistische_regression(X_train_s, y_train)
test_scores_numpy = sigmoid(X_test_s @ w_numpy + b_numpy)
test_pred_numpy = (test_scores_numpy >= 0.5).astype(int)

print("Gewichte:", np.round(w_numpy, 3))
print("Bias:", round(b_numpy, 3))
print(f"Testgenauigkeit: {accuracy_score(y_test, test_pred_numpy):.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(historie_numpy["Epoche"], historie_numpy["Kreuzentropie"])
ax.set_title("Training der logistischen NumPy-Regression")
ax.set_xlabel("Epoche")
ax.set_ylabel("Kreuzentropie")
plt.show()

> **Musterantwort und Interpretation**
>
> Ähnliche Größenordnungen der Merkmale führen meist zu ausgeglicheneren Gradienten und erlauben eine gemeinsame Lernrate. Ohne Skalierung kann ein großes Merkmal die Updates dominieren und die Optimierung langsam oder instabil machen.

### Aufgabe 3: Klassifikatoren und Baseline fair vergleichen

Vergleichen Sie auf demselben Split:

- `DummyClassifier(strategy="most_frequent")`,
- scikit-learn `LogisticRegression`,
- `KNeighborsClassifier(n_neighbors=5)`,
- `GaussianNB`,
- Ihre NumPy-logistische Regression.

Verwenden Sie dieselben skalierten Daten, berechnen Sie Accuracy und dokumentieren Sie eine wichtige Annahme oder Grenze jedes Verfahrens.

In [ ]:
# X_train_s, X_test_s, y_train, y_test und test_pred_numpy sind vorhanden.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

modelle = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistische Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "k-NN": KNeighborsClassifier(n_neighbors=5),
    "Gaussian Naive Bayes": GaussianNB(),
}

ergebniszeilen = []
for name, modell in modelle.items():
    modell.fit(X_train_s, y_train)
    pred = modell.predict(X_test_s)
    ergebniszeilen.append({"Modell": name, "Accuracy": accuracy_score(y_test, pred)})

ergebniszeilen.append(
    {"Modell": "NumPy logistische Regression", "Accuracy": accuracy_score(y_test, test_pred_numpy)}
)

modellvergleich = pd.DataFrame(ergebniszeilen).sort_values("Accuracy", ascending=False)
display(modellvergleich.round(3))

> **Musterantwort und Interpretation**
>
> Die logistische Regression bildet eine lineare Entscheidungsgrenze im Merkmalsraum. k-NN hängt stark von Skalierung, Distanzmaß und lokaler Datendichte ab. Gaussian Naive Bayes nimmt klassenweise näherungsweise normalverteilte Merkmale und bedingte Unabhängigkeit an. Gute Testergebnisse ersetzen keine Prüfung dieser Annahmen.

### Aufgabe 4: Eine K-Means-Iteration manuell durchführen

Verwenden Sie die ersten zwölf Punkte von `X_cluster` und drei vorgegebene Startzentren:

1. Berechnen Sie die quadratischen euklidischen Distanzen jedes Punkts zu jedem Zentrum.
2. Ordnen Sie jeden Punkt dem nächsten Zentrum zu.
3. Berechnen Sie neue Zentren als Mittelwert ihrer zugeordneten Punkte.
4. Berechnen Sie die Inertia vor und nach der Aktualisierung.
5. Behandeln Sie den Sonderfall eines leeren Clusters nachvollziehbar.

In [ ]:
X_kmeans_klein = X_cluster[:12]
zentren_start = np.array([[-3.5, -2.5], [0.5, 2.5], [2.5, -0.5]], dtype=float)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Broadcasting erzeugt die Form (Punkte, Zentren, Merkmale).
differenzen = X_kmeans_klein[:, np.newaxis, :] - zentren_start[np.newaxis, :, :]
distanzen_quadrat = np.sum(differenzen ** 2, axis=2)
zuordnung = np.argmin(distanzen_quadrat, axis=1)

neue_zentren = []
for cluster_id in range(len(zentren_start)):
    cluster_punkte = X_kmeans_klein[zuordnung == cluster_id]
    if len(cluster_punkte) == 0:
        # Ein leeres Cluster behält hier sein altes Zentrum. Andere Strategien wären möglich.
        neues_zentrum = zentren_start[cluster_id]
    else:
        neues_zentrum = cluster_punkte.mean(axis=0)
    neue_zentren.append(neues_zentrum)
neue_zentren = np.vstack(neue_zentren)

# Inertia ist die Summe der quadratischen Abstände zum jeweils zugeordneten Zentrum.
inertia_vorher = np.sum(distanzen_quadrat[np.arange(len(X_kmeans_klein)), zuordnung])
neue_diff = X_kmeans_klein[:, np.newaxis, :] - neue_zentren[np.newaxis, :, :]
neue_dist_q = np.sum(neue_diff ** 2, axis=2)
inertia_nachher = np.sum(neue_dist_q[np.arange(len(X_kmeans_klein)), zuordnung])

print("Zuordnung:", zuordnung)
print("Startzentren:\n", np.round(zentren_start, 3))
print("Neue Zentren:\n", np.round(neue_zentren, 3))
print(f"Inertia vorher: {inertia_vorher:.3f}")
print(f"Inertia nach Zentrenupdate: {inertia_nachher:.3f}")
assert inertia_nachher <= inertia_vorher + 1e-10

> **Musterantwort und Interpretation**
>
> K-Means kann abhängig von Startzentren in einem lokalen Minimum enden. Außerdem optimiert es nur kompakte quadratische Distanzen zu Mittelwertzentren. Nicht kugelförmige Cluster, Ausreißer, unpassend skalierte Merkmale oder fachlich irrelevante Gruppen können trotz Konvergenz problematisch bleiben.

### Aufgabe 5: Clusterzahl, Skalierung und Gegenbeispiel prüfen

1. Standardisieren Sie `X_cluster` und vergleichen Sie K-Means für k = 2 bis 6 mit Inertia und Silhouette.
2. Wählen Sie eine begründete Clusterzahl.
3. Erzeugen Sie `make_moons` und wenden Sie K-Means mit k = 2 an.
4. Visualisieren Sie beide Ergebnisse und erklären Sie, warum K-Means bei gekrümmten Gruppen Schwierigkeiten hat.

In [ ]:
cluster_scaler = StandardScaler()
X_cluster_s = cluster_scaler.fit_transform(X_cluster)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

vergleich_clusterzahl = []
for k in range(2, 7):
    modell = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = modell.fit_predict(X_cluster_s)
    vergleich_clusterzahl.append(
        {"k": k, "Inertia": modell.inertia_, "Silhouette": silhouette_score(X_cluster_s, labels)}
    )

clusterzahl_df = pd.DataFrame(vergleich_clusterzahl)
display(clusterzahl_df.round(3))

bestes_k = int(clusterzahl_df.loc[clusterzahl_df["Silhouette"].idxmax(), "k"])
bestes_kmeans = KMeans(n_clusters=bestes_k, random_state=RANDOM_SEED, n_init=10)
beste_labels = bestes_kmeans.fit_predict(X_cluster_s)

X_moons, y_moons_true = make_moons(n_samples=220, noise=0.07, random_state=RANDOM_SEED)
moons_labels = KMeans(n_clusters=2, random_state=RANDOM_SEED, n_init=10).fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_cluster_s[:, 0], X_cluster_s[:, 1], c=beste_labels, alpha=0.75)
axes[0].set_title(f"K-Means auf kompakten Gruppen, k={bestes_k}")
axes[0].set_xlabel("Merkmal 1, skaliert")
axes[0].set_ylabel("Merkmal 2, skaliert")

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=moons_labels, alpha=0.75)
axes[1].set_title("K-Means auf gekrümmten Monden")
axes[1].set_xlabel("Merkmal 1")
axes[1].set_ylabel("Merkmal 2")
plt.tight_layout()
plt.show()

> **Musterantwort und Interpretation**
>
> Inertia sinkt fast immer, wenn mehr Cluster zugelassen werden, und erreicht bei einem Cluster pro Punkt theoretisch null. Die Wahl muss daher zusätzlich Trennung, Stabilität, Fachbedeutung und praktischen Nutzen berücksichtigen. Silhouette ist hilfreich, aber ebenfalls keine fachliche Garantie.

### Aufgabe 6: PCA mit NumPy berechnen und erklärte Varianz prüfen

1. Standardisieren Sie die vier Iris-Merkmale.
2. Berechnen Sie Kovarianzmatrix, Eigenwerte und Eigenvektoren mit NumPy.
3. Sortieren Sie Komponenten absteigend nach Eigenwert.
4. Projizieren Sie auf zwei Hauptkomponenten.
5. Berechnen Sie erklärte Varianzanteile und kumulierten Anteil.
6. Visualisieren Sie die Projektion nach echter Iris-Klasse, ohne die Labels bei der PCA-Berechnung zu verwenden.

In [ ]:
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Standardisierung verhindert, dass Merkmale mit größerer Skala die Kovarianz dominieren.
iris_scaler = StandardScaler()
X_iris_s = iris_scaler.fit_transform(X_iris)

kovarianz = np.cov(X_iris_s, rowvar=False)
eigenwerte, eigenvektoren = np.linalg.eigh(kovarianz)

# eigh liefert aufsteigende Eigenwerte, deshalb sortieren wir absteigend.
reihenfolge = np.argsort(eigenwerte)[::-1]
eigenwerte = eigenwerte[reihenfolge]
eigenvektoren = eigenvektoren[:, reihenfolge]

projektion_2d = X_iris_s @ eigenvektoren[:, :2]
erklaerte_anteile = eigenwerte / eigenwerte.sum()

print("Eigenwerte:", np.round(eigenwerte, 3))
print("Erklärte Varianzanteile:", np.round(erklaerte_anteile, 3))
print("Kumuliert nach zwei Komponenten:", round(erklaerte_anteile[:2].sum(), 3))
print("Projektionsform:", projektion_2d.shape)

fig, ax = plt.subplots(figsize=(7, 5))
for klasse, name in enumerate(iris.target_names):
    maske = y_iris == klasse
    ax.scatter(projektion_2d[maske, 0], projektion_2d[maske, 1], label=name, alpha=0.75)
ax.set_title("Iris-Daten in zwei Hauptkomponenten")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Jede Hauptkomponente ist eine gewichtete Mischung mehrerer Originalmerkmale. Sie maximiert Varianz, besitzt aber nicht automatisch eine direkte fachliche Einheit oder Bedeutung. Die Ladungen können Hinweise geben, dennoch bleibt die Erklärung indirekter als bei Originalspalten.

### Aufgabe 7: Integrationsaufgabe: PCA, Clustering und Anomaliehinweis

Erweitern Sie `X_cluster` um drei weit entfernte Punkte. Führen Sie anschließend einen Workflow aus:

1. Standardisierung.
2. PCA auf zwei Komponenten.
3. K-Means mit der zuvor gewählten Clusterzahl.
4. Anomaliehinweis über den Abstand zum zugeordneten Zentrum: markieren Sie die obersten 3 %.
5. Erstellen Sie eine Tabelle mit Cluster, Zentrumabstand und Anomalieflag.
6. Visualisieren und interpretieren Sie die Grenzen dieser einfachen Anomalieregel.

In [ ]:
X_erweitert = np.vstack([X_cluster, [[8, 8], [-8, 7], [7, -8]]])

# ============================================================
# MUSTERLÖSUNG
# ============================================================

scaler_int = StandardScaler()
X_erweitert_s = scaler_int.fit_transform(X_erweitert)

# Bei zwei Originalmerkmalen bleibt die PCA zwar zweidimensional, dreht aber die Achsen in Varianzrichtung.
kov_int = np.cov(X_erweitert_s, rowvar=False)
werte_int, vektoren_int = np.linalg.eigh(kov_int)
ordnung_int = np.argsort(werte_int)[::-1]
X_pca_int = X_erweitert_s @ vektoren_int[:, ordnung_int[:2]]

kmeans_int = KMeans(n_clusters=bestes_k, random_state=RANDOM_SEED, n_init=10)
cluster_int = kmeans_int.fit_predict(X_pca_int)
zentren_int = kmeans_int.cluster_centers_[cluster_int]
abstaende = np.linalg.norm(X_pca_int - zentren_int, axis=1)
schwelle = np.quantile(abstaende, 0.97)
anomalieflag = abstaende >= schwelle

integrationsbericht = pd.DataFrame(
    {
        "Cluster": cluster_int,
        "Zentrumabstand": abstaende,
        "Anomalie": anomalieflag,
    }
)
print("Markierte Punkte:", int(anomalieflag.sum()))
display(integrationsbericht.nlargest(8, "Zentrumabstand"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_pca_int[:, 0], X_pca_int[:, 1], c=cluster_int, alpha=0.65)
ax.scatter(
    X_pca_int[anomalieflag, 0],
    X_pca_int[anomalieflag, 1],
    marker="x",
    s=130,
    linewidths=2,
    label="oberste 3 % Zentrumabstand",
)
ax.set_title("PCA-Projektion mit Cluster- und Anomaliehinweisen")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Sie kann lokale Anomalien übersehen, die innerhalb eines großen oder länglichen Clusters relativ zentrumsnah liegen, aber in ihrer Nachbarschaft ungewöhnlich sind. Außerdem hängt sie stark von K-Means, Skalierung, gewählter Clusterzahl und dem willkürlichen 3-Prozent-Schwellenwert ab. Fachwissen und alternative Dichtemethoden sind erforderlich.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?